## 0. Compute all city centers.

In [ ]:
import pandas as pd
import os, glob
import numpy as np

In [ ]:
files = glob.glob("../../data/datasets/sv_paths/*.pkl")
len(files)

In [ ]:
df_list = []
for f in files:
    tmp = pd.read_pickle(f)
    df_list.append(tmp)
df = pd.concat(df_list, ignore_index=True)
df

In [ ]:
df['panoid'] = df['path'].apply(lambda x: x.split("/")[-1].replace(".jpg", "").rsplit("_", 1)[0])
df

In [ ]:
df.drop_duplicates(subset=["panoid"], inplace=True)
df

In [ ]:
df['country'] = df["path"].str.split("/").str[4]
df['city'] = df["path"].str.split("/").str[5]
df

In [ ]:
del df['path'], df['exists'], df['geometry']
df

In [ ]:
print(df["city"].nunique())
df["city"].unique()

In [ ]:
df['lon'].isna().sum()

In [ ]:
df[df['lon'].isna()]

In [ ]:
# 1. Unique (country, city) pairs to process.
pairs_to_process = df[df['lon'].isna()][['country','city']].drop_duplicates().itertuples(index=False)

print(f"Found {len(list(pairs_to_process))} (country, city) pairs with missing lon/lat.")
# Re-iterate (itertuples is a one-shot iterator).
pairs_to_process = df[df['lon'].isna()][['country','city']].drop_duplicates().itertuples(index=False)

# 2. Iterate over the pairs.
for country, city in pairs_to_process:
    print(f"Processing {country}, {city} ...")
    
    try:
        # 3. Load the corresponding metadata file.
        meta_path = f"../../data/raw/GoogleSV/metadata/{country}/{city}/panoids.csv"
        meta = pd.read_csv(meta_path)
        
        # 4. Build a lookup map.
        # Key step: set panoid as the index for fast lookups.
        # drop_duplicates() ensures panoid is unique in case of dups.
        lon_map = meta.drop_duplicates(subset='panoid').set_index('panoid')['lon']
        lat_map = meta.drop_duplicates(subset='panoid').set_index('panoid')['lat']
        
        # 5. Identify rows in the main df that need updating.
        # Only update rows in the current country/city where lon is NaN.
        mask = (df['country'] == country) & (df['city'] == city) & (df['lon'].isna())
        
        # 6. Fill values efficiently with .map().
        # .map() looks up lon/lat from lon_map/lat_map by panoid.
        # If a panoid isn't in the map (the previous error case),
        # .map() returns NaN automatically instead of erroring.
        df.loc[mask, 'lon'] = df.loc[mask, 'panoid'].map(lon_map)
        df.loc[mask, 'lat'] = df.loc[mask, 'panoid'].map(lat_map)

    except FileNotFoundError:
        print(f"  WARNING: Metadata file not found at {meta_path}. Skipping.")
    except KeyError:
        # If panoids.csv lacks 'panoid', 'lon', or 'lat' columns.
        print(f"  WARNING: File {meta_path} is missing expected columns. Skipping.")
    except Exception as e:
        print(f"  ERROR processing {country}, {city}: {e}")

# 7. Check how many NaNs remain.
final_nan_count = df['lon'].isna().sum()
print(f"---")
print(f"Processing complete. Total remaining NaNs in 'lon': {final_nan_count}")

In [ ]:
df.dropna(subset=['lon', 'lat'], inplace=True)
df.reset_index(drop=True, inplace=True)
df

In [ ]:
city_df = df[['city', 'lat', 'lon']].groupby('city').mean()
city_df.reset_index(inplace=True)
city_df

In [ ]:
city_df = city_df.merge(df[['city', 'country']].drop_duplicates(), on='city', how='left')
city_df

In [ ]:
city_df.sort_values(by=['country', 'city'], inplace=True)
city_df.reset_index(drop=True, inplace=True)
city_df

In [ ]:
city_df.to_csv("../../data/processed/fig/all_city_center.csv", index=False)

## 1. Number of indicators per country.

In [ ]:
import pandas as pd
import geopandas as gpd
import os, glob

In [ ]:
files = glob.glob("../../data/processed/0labels/*.csv")
len(files)

In [ ]:
df_list = []
for f in files:
    tmp = pd.read_csv(f)
    tmp['country'] = os.path.basename(f).split(".")[0]
    df_list.append(tmp)
df = pd.concat(df_list, ignore_index=False)
df

In [ ]:
country_df = df.groupby('country').nunique()
country_df.reset_index(inplace=True)
country_df

In [ ]:
world = gpd.read_file('../../data/processed/world.geojson')
world

In [ ]:
china_geo = world[(world['name'] == 'Taiwan') | (world['name'] == 'China')].unary_union
world.loc[31, 'geometry'] = china_geo
world = world[world['name'] != 'Taiwan']
world['name'] = world['name'].apply(lambda x: 'US' if x == 'United States of America' else x)
world

In [ ]:
country_df = country_df.merge(world[['name', 'geometry']], left_on='country', right_on='name', how='left')
country_df

In [ ]:
country_df = gpd.GeoDataFrame(country_df, geometry='geometry')
country_df.to_file("../../data/processed/fig/extended_fig1_country_sdg.shp")